# 03 - LoRA persona DPO, evaluation generations, blinded pair files  (Tasks 5, 6)

Inputs: **attach the outputs of notebooks 01 and 02.**

1. Task 5: TRL `DPOTrainer` + LoRA on `HuggingFaceTB/SmolLM2-360M-Instruct` with the Ramsay persona pairs (~5-10 min).
2. Task 6: a 3-prompt sanity check for every checkpoint, then full generation with identical decoding
   (greedy, repetition_penalty 1.1; 512 new tokens, 256 for persona), split over both GPUs:
   * AlpacaEval-300 (seed 42): SFT and UltraFeedback-DPO
   * HH eval set (300): SFT and HH-RLHF-DPO
   * Persona eval set (40): SmolLM2-Instruct base and LoRA-DPO
3. Blinded pair files (random A/B order, hidden model mapping) for DPO-vs-SFT and LoRA-vs-base.
4. Copies run metrics/configs (no checkpoints) so notebook 04 has everything needed for the report tables.

**Kaggle settings for this notebook:** Accelerator = **GPU T4 x2**, Internet = **On** (needs a phone-verified account),
Persistence = Files only (optional). Edit `REPO_URL` below, then **Save Version -> Save & Run All (Commit)** so the
run continues in the background and all files in `/kaggle/working` are kept as this version's output.

In [ ]:
# ---- EDIT THIS: your GitHub fork of the starter repo with this code pushed to it ----
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/cs5326-pa2"

BUILD_OPTIONAL_SFT_VS_BASE = True   # extra pair file (SFT vs base on AlpacaEval) for your own analysis

In [ ]:
import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
from smalign.notebook import sh, start_bg, wait_bg, find_input, copy_run_metadata, zip_dir, disk_report

In [ ]:
# Task 1 + Task 5 deps: same as above plus TRL/PEFT (trl 0.29.1 = newest release pinned to transformers<5)
sh("python -m pip uninstall -y -q torchao || true")
sh('python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml '
   'sentencepiece safetensors pytest matplotlib "trl==0.29.1" "peft>=0.13"')
sh("python -m pip install -q -e . --no-deps")

In [ ]:
# Task 1: confirm two T4 GPUs and print package versions
sh("nvidia-smi")
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if torch.cuda.device_count() < 2:
    print("WARNING: expected GPU T4 x2 (Settings -> Accelerator). Scripts will fall back to one GPU but run slower.")
disk_report()

import trl, peft
print("trl", trl.__version__, "| peft", peft.__version__)

## Task 5 - LoRA persona DPO (`scripts/train_lora_dpo.py`, `configs/lora_persona.yaml`)

In [ ]:
if not os.path.exists("outputs/lora_persona/adapter/adapter_config.json"):
    sh("CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_dpo.py --config configs/lora_persona.yaml", log_path="outputs/logs/lora_persona.log")
sh("cat outputs/lora_persona/summary.json | head -c 1500")

## Locate the checkpoints from notebooks 01 and 02

In [ ]:
SFT_BEST = os.path.dirname(find_input("outputs/sft/best/config.json"))
DPO_HH = os.path.dirname(find_input("outputs/dpo_hh/best/config.json"))
DPO_UF = os.path.dirname(find_input("outputs/dpo_uf/best/config.json"))
SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"
LORA = "outputs/lora_persona/adapter"
ALPACA, HH, PERSONA = "eval_sets/alpaca_eval_300_seed42.json", "eval_sets/hh_rlhf_eval_300_seed42.json", "eval_sets/persona_style_eval_40.json"

def gen_cmd(model, name, eval_set, out, device, extra=""):
    return (f"python scripts/generate.py --model_path {model} --generator {name} --eval_set {eval_set} "
            f"--output_path {out} --device {device} {extra}")

## Task 6 sanity check: 3 prompts per checkpoint, inspected manually before the full run

In [ ]:
sanity = [
    (SFT_BEST, "sft", ALPACA, ""), (DPO_UF, "dpo_uf", ALPACA, ""), (DPO_HH, "dpo_hh", HH, ""),
    (SMOL, "lora_dpo", PERSONA, f"--adapter_path {LORA} --template chat_template --max_new_tokens 256"),
]
for model, name, eval_set, extra in sanity:
    sh(gen_cmd(model, name, eval_set, f"/tmp/sanity_{name}.json", "cuda:0", extra + " --limit 3 --print_samples"))

## Full generation (two GPU queues in parallel; files are written after every batch and resumable)

In [ ]:
G = "outputs/generations"
queue0 = [
    gen_cmd(SFT_BEST, "sft", ALPACA, f"{G}/sft_alpaca.json", "cuda:0"),
    gen_cmd(DPO_UF, "dpo_uf", ALPACA, f"{G}/dpo_uf_alpaca.json", "cuda:0"),
    gen_cmd(SMOL, "smollm2_instruct_base", PERSONA, f"{G}/smollm_base_persona.json", "cuda:0", "--template chat_template --max_new_tokens 256"),
]
queue1 = [
    gen_cmd(SFT_BEST, "sft", HH, f"{G}/sft_hh.json", "cuda:1"),
    gen_cmd(DPO_HH, "dpo_hh", HH, f"{G}/dpo_hh_hh.json", "cuda:1"),
    gen_cmd(SMOL, "lora_dpo", PERSONA, f"{G}/lora_dpo_persona.json", "cuda:1", f"--adapter_path {LORA} --template chat_template --max_new_tokens 256"),
]
jobs = [start_bg(" && ".join(queue0), "outputs/logs/generation_gpu0.log"),
        start_bg(" && ".join(queue1), "outputs/logs/generation_gpu1.log")]
wait_bg(*jobs, poll_seconds=180)
sh(f"ls -la {G}")

## Blinded pairwise files (judge sees only Response A / Response B)

In [ ]:
E = "outputs/eval"
sh(f"python scripts/build_pairs.py --candidate {G}/dpo_uf_alpaca.json --baseline {G}/sft_alpaca.json --rubric general --output_path {E}/dpo_uf_vs_sft_pairs.jsonl")
sh(f"python scripts/build_pairs.py --candidate {G}/dpo_hh_hh.json --baseline {G}/sft_hh.json --rubric safety --eval_set {HH} --output_path {E}/dpo_hh_vs_sft_pairs.jsonl")
sh(f"python scripts/build_pairs.py --candidate {G}/lora_dpo_persona.json --baseline {G}/smollm_base_persona.json --rubric persona --output_path {E}/lora_dpo_vs_base_pairs.jsonl")

# baseline generations from notebook 01 (for analysis) + optional SFT-vs-base pairs
import shutil, glob
for path in glob.glob("/kaggle/input/**/outputs/generations/base_*.json", recursive=True):
    shutil.copy2(path, G)
if BUILD_OPTIONAL_SFT_VS_BASE and os.path.exists(f"{G}/base_alpaca.json"):
    os.makedirs(f"{E}/optional", exist_ok=True)
    sh(f"python scripts/build_pairs.py --candidate {G}/sft_alpaca.json --baseline {G}/base_alpaca.json --rubric general --output_path {E}/optional/sft_vs_base_pairs.jsonl")

## Collect run metadata (metrics, configs, summaries; no checkpoints) for the report

In [ ]:
for run in ("sft", "dpo_hh", "dpo_uf"):
    copy_run_metadata(os.path.dirname(find_input(f"outputs/{run}/summary.json")), f"outputs/{run}")
for path in glob.glob("/kaggle/input/**/outputs/data/data_stats.json", recursive=True)[:1]:
    os.makedirs("outputs/data", exist_ok=True); shutil.copy2(path, "outputs/data/data_stats.json")
for path in glob.glob("/kaggle/input/**/outputs/logs/*.log", recursive=True):
    os.makedirs("outputs/logs", exist_ok=True); shutil.copy2(path, "outputs/logs/")

sh("python scripts/plot_training.py --sft outputs/sft --dpo outputs/dpo_hh outputs/dpo_uf --lora outputs/lora_persona --out_dir outputs/report_assets")
sh("python scripts/summarize_results.py")
from IPython.display import Image, display
display(Image("outputs/report_assets/lora_persona_curves.png"))
zip_dir("outputs", "/kaggle/working/generation_outputs.zip")
print("DONE. Save this version; attach its output to notebook 04 (the judge notebook).")